<a href="https://colab.research.google.com/github/slinkp/fastai-study-group-2026/blob/main/course22/clean/02-saving-a-basic-fastai-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Saving a Cats v Dogs Model

In [1]:
# Make sure we've got the latest version of fastai:
!pip install -Uqq fastai

In [3]:
from fastai.vision.all import (
    untar_data, ImageDataLoaders,
    get_image_files, vision_learner, resnet18,
    Resize,
    error_rate,
    URLs
)

In [4]:
path = untar_data(URLs.PETS)/'images'

<div><progress max="811706944" value="811712512"></progress> 100.00% [811712512/811706944 00:21&lt;00:00]</div>

In [5]:
def is_cat(x):
    return x[0].isupper()

In [6]:
dls = ImageDataLoaders.from_name_func('.',
    get_image_files(path), valid_pct=0.2, seed=42,
    label_func=is_cat,
    item_tfms=Resize(192))

In [7]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


epoch,train_loss,valid_loss,error_rate,time
0,0.196835,0.039920,0.014885,00:38


epoch,train_loss,valid_loss,error_rate,time
0,0.071697,0.022397,0.009472,00:37
1,0.051864,0.016365,0.006766,00:37
2,0.025246,0.016692,0.005413,00:35


In [8]:
import torch

In [9]:
#learn.export('model.pkl')
## SAFER THAN PICKLE:
torch.save(learn.model.state_dict(), 'cats_dogs_model_compressed.pth',
           _use_new_zipfile_serialization=True)


Apparently torch.save still uses pickle under the hood? This approach just serializes only the dict. So we're still expecting the user of our saved model to trust our file.

ONNX was supposed to be safer than pickle, but there are [still attack vectors!](https://ai-alert.org/posts/model-file-format-vulnerabilities-pickle-onnx-safetensors/#onnx-a-different-risk-profile)

The new hotness is [SafeTensors](https://huggingface.co/docs/safetensors/index)

In [10]:
pip install safetensors

In [12]:
from safetensors.torch import save_file

save_file(learn.model.state_dict(), "cats_dogs.safetensors")

In [14]:
# Let's try loading that

from safetensors import safe_open

with safe_open("cats_dogs.safetensors", framework="pt", device=0) as f:
    tensors = {key: f.get_tensor(key) for key in f.keys()}

In [15]:
# How do we load that into a model?
# Aha, we don't actually use tensors directly like that.

from safetensors.torch import load_file
learn2 = load_file("cats_dogs.safetensors")
learn2

{'0.1.num_batches_tracked': tensor(368),
 '0.4.0.bn1.num_batches_tracked': tensor(368),
 '0.4.0.bn2.num_batches_tracked': tensor(368),
 '0.4.1.bn1.num_batches_tracked': tensor(368),
 '0.4.1.bn2.num_batches_tracked': tensor(368),
 '0.5.0.bn1.num_batches_tracked': tensor(368),
 '0.5.0.bn2.num_batches_tracked': tensor(368),
 '0.5.0.downsample.1.num_batches_tracked': tensor(368),
 '0.5.1.bn1.num_batches_tracked': tensor(368),
 '0.5.1.bn2.num_batches_tracked': tensor(368),
 '0.6.0.bn1.num_batches_tracked': tensor(368),
 '0.6.0.bn2.num_batches_tracked': tensor(368),
 '0.6.0.downsample.1.num_batches_tracked': tensor(368),
 '0.6.1.bn1.num_batches_tracked': tensor(368),
 '0.6.1.bn2.num_batches_tracked': tensor(368),
 '0.7.0.bn1.num_batches_tracked': tensor(368),
 '0.7.0.bn2.num_batches_tracked': tensor(368),
 '0.7.0.downsample.1.num_batches_tracked': tensor(368),
 '0.7.1.bn1.num_batches_tracked': tensor(368),
 '0.7.1.bn2.num_batches_tracked': tensor(368),
 '1.2.num_batches_tracked': tensor(368)